In [99]:
import sys
from pathlib import Path
from pyprojroot import here

sys.path.append(str(here()))

In [100]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils import load_data
from sklearn.model_selection import train_test_split
import torch
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
import importlib
from src.config import cfg
from src.processing import (
    FinalTreeTransformer,
    MemoryOptimizer,
    new_num_columns,
    new_ohe_columns,
    new_ordinal_categories,
    new_ordinal_columns,
)

from src.utils import set_seed

In [101]:
import warnings

warnings.filterwarnings("ignore", message="Found unknown categories in columns")

In [102]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [103]:
gseed = cfg.general.seed
set_seed(gseed)

In [104]:
train_path = Path(cfg.paths.train)
test_path = Path(cfg.paths.test)

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)
df_all_data = pd.concat(
    [df_train.drop(columns=["SalePrice"]), df_test], axis=0
).reset_index(drop=True)

In [105]:
X_train, y_train_raw, X_test, test_ids = load_data(cfg, False)

y_train = np.log1p(y_train_raw)

In [106]:
import src.nn as src_nn

importlib.reload(src_nn)

final_lin = src_nn.final_lin


In [107]:
from sklearn.base import clone
from sklearn.inspection import permutation_importance
from sklearn.metrics import root_mean_squared_error

In [108]:
def prepare_early_stopping_params(model, X_val, y_val, early_stopping_callback=None):
    model_name = type(model).__name__
    fit_params = {}

    if "LGBM" in model_name:
        fit_params["model__eval_X"] = X_val
        fit_params["model__eval_y"] = y_val

        if early_stopping_callback is not None:
            fit_params["model__callbacks"] = [early_stopping_callback]
        else:
            import lightgbm as lgb

            fit_params["model__callbacks"] = [
                lgb.early_stopping(stopping_rounds=100, verbose=False)
            ]

    elif "XGB" in model_name:
        fit_params["model__eval_set"] = [(X_val, y_val)]

    elif "CatBoost" in model_name:
        fit_params["model__eval_set"] = (X_val, y_val)
        fit_params["model__early_stopping_rounds"] = 100
        fit_params["model__verbose"] = False

    elif "HistGradientBoosting" in model_name:
        fit_params["model__X_val"] = X_val
        fit_params["model__y_val"] = y_val

    elif "GradientBoosting" in model_name:
        pass

    elif "MLP" in model_name:
        fit_params["model__eval_X"] = X_val
        fit_params["model__eval_y"] = y_val

    return fit_params


def cv_result(
    model,
    X_train,
    y_train,
    y_strat,
    cv_splitter,
    preprocessor,
    name=None,
    fit_params=None,
    use_early_stopping=False,
    early_stopping_callback=None,
    calculate_importance=False,
    perm_n_repeats=5,
    random_state=gseed,
):
    """
    Universal cv function, but could use LGBM early stopping
    """
    fit_params = fit_params or {}
    model_pipe = Pipeline([("preprocessor", preprocessor), ("model", model)])

    fitted_models = []
    oof_preds_sum = np.zeros(len(X_train), dtype=float)
    oof_counts = np.zeros(len(X_train), dtype=float)

    tr_scores, val_scores = [], []

    fold_importances = []

    splits = list(cv_splitter.split(X_train, y_strat))

    for train_idx, val_idx in tqdm(
        splits, desc=name or "CV folds", unit="fold", leave=False
    ):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        fold_pipe = clone(model_pipe)

        fold_fit_params = fit_params.copy()

        if use_early_stopping:
            fold_preprocessor = clone(preprocessor)
            X_tr_trans = fold_preprocessor.fit_transform(X_tr, y_tr)
            X_val_trans = fold_preprocessor.transform(X_val)

            framework_params = prepare_early_stopping_params(
                model=model,
                X_val=X_val_trans,
                y_val=y_val,
                early_stopping_callback=early_stopping_callback,
            )
            fold_fit_params.update(framework_params)

        fold_pipe.fit(X_tr, y_tr, **fold_fit_params)

        if calculate_importance:
            res = permutation_importance(
                fold_pipe,
                X_val,
                y_val,
                scoring="neg_root_mean_squared_error",
                n_repeats=perm_n_repeats,
                random_state=random_state,
                n_jobs=-1,
            )
            fold_importances.append(res.importances_mean)

        fold_preds_tr = fold_pipe.predict(X_tr)
        fold_preds_val = fold_pipe.predict(X_val)

        oof_preds_sum[val_idx] += fold_preds_val
        oof_counts[val_idx] += 1

        fitted_models.append(fold_pipe)

        tr_scores.append(root_mean_squared_error(y_tr, fold_preds_tr))
        val_scores.append(root_mean_squared_error(y_val, fold_preds_val))

    oof_preds = oof_preds_sum / oof_counts
    oof_mse = root_mean_squared_error(y_train, oof_preds)

    metrics = pd.DataFrame(
        [
            {
                "model": name,
                "TRAIN_rmsle_MEAN": np.mean(tr_scores),
                "TRAIN_rmsle_STD": np.std(tr_scores),
                "VAL_rmsle_MEAN": np.mean(val_scores),
                "VAL_rmsle_STD": np.std(val_scores),
                "OOF_rmsle": oof_mse,
            }
        ]
    )

    importance_df = None
    if calculate_importance:
        importance_df = pd.DataFrame(fold_importances, columns=X_train.columns).T
        fold_cols = importance_df.columns.tolist()

        importance_df["importance_mean"] = importance_df[fold_cols].mean(axis=1)
        importance_df["importance_std"] = importance_df[fold_cols].std(axis=1)

        importance_df = importance_df.sort_values(by="importance_mean", ascending=False)

        return fitted_models, oof_preds, metrics, val_scores, importance_df
    return fitted_models, oof_preds, metrics, val_scores

In [109]:
from sklearn.model_selection import RepeatedStratifiedKFold

In [110]:
model = src_nn.MLPRegressor(lr=0.1)
y_binned = pd.qcut(y_train, q=10, labels=False)

rskf = RepeatedStratifiedKFold(n_splits=10, n_repeats=5, random_state=gseed)

In [111]:
fitted_models, oof_preds, metrics, val_scores = cv_result(
    model=model,
    X_train=X_train,
    y_train=y_train,
    y_strat=y_binned,
    cv_splitter=rskf,
    preprocessor=final_lin,
    name="nn",
    use_early_stopping=True,
)

nn:   0%|          | 0/50 [00:00<?, ?fold/s]

nn:   0%|          | 0/500 [00:00<?, ?it/s]

[Early Stopping] Обучение остановлено на эпохе 224


nn:   0%|          | 0/500 [00:00<?, ?it/s]

[Early Stopping] Обучение остановлено на эпохе 106


...

[Early Stopping] Обучение остановлено на эпохе 126


nn:   0%|          | 0/500 [00:00<?, ?it/s]

[Early Stopping] Обучение остановлено на эпохе 111


nn:   0%|          | 0/500 [00:00<?, ?it/s]

[Early Stopping] Обучение остановлено на эпохе 97


In [112]:
metrics

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,nn,0.068584,0.016698,0.124447,0.014563,0.111752
